<a href="https://colab.research.google.com/github/erdilix/nma-compneu2026/blob/main/projects/behavior_and_theory/earth_laquitaine_fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fitting Bayesian observer models to real Laquitaine data

Companion to `earth_laquitaine.ipynb`. There the three models (static, static-with-warm-up, online) were *generative* — they output one MAP estimate per trial. To **fit them to real human data** we need a likelihood `P(estimate | stimulus, params)`, so here each model is made **probabilistic**:

- **von Mises** prior and likelihood (directions are circular),
- the latent measurement is **marginalized out** (W3D2 Tutorial 3),
- a **lapse** term removes zero-probabilities.

Then we fit free parameters by minimizing the **negative log likelihood (NLL)** and compare the three models with **AIC** — the static-vs-online test from the Laquitaine template.

**Plan:** load data → build the von Mises observer → recover parameters from *synthetic* data (sanity check) → fit all three models to one subject/one prior block → compare AIC.

In [ ]:
# @title Dependencies + data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import vonmises  # noqa: F401 (kept for reference)
from scipy.optimize import minimize
import os, requests

URL = "https://github.com/steevelaquitaine/projInference/raw/gh-pages/data/csv/data01_direction4priors.csv"
CSV = "data01_direction4priors.csv"
if not os.path.exists(CSV):
    r = requests.get(URL)
    assert r.status_code == 200, "download failed; contact steeve.laquitaine@epfl.ch"
    open(CSV, "wb").write(r.content)

data = pd.read_csv(CSV)
print(data.shape)
data[["subject_id", "prior_std", "motion_coherence", "motion_direction"]].head()

## The von Mises Bayesian observer

Everything lives on a coarse direction grid (`GRID_STEP` deg) to keep fitting fast — shrink the step for accuracy, grow it for speed.

- `kernel(kappa)` — an `n×n` matrix whose row *i* is a von Mises centred on grid point *i*. Because von Mises depends only on the angular difference, this single matrix serves as **both** the likelihood (given a measurement) and `P(measurement | stimulus)`.
- `response_prob` — for a fixed prior, each possible measurement yields a deterministic MAP response; we sum `P(measurement | stimulus)` over all measurements landing on each response, then mix in the lapse. This is the marginalization step.

In [ ]:
# @title Grid + observer building blocks
GRID_STEP = 4                      # deg; smaller = more accurate, slower
xg = np.arange(1, 361, GRID_STEP)  # direction grid
n = len(xg)

def to_idx(val):
    return np.clip(np.round((np.asarray(val, float) - xg[0]) / GRID_STEP).astype(int), 0, n - 1)

def sig(z):    return 1.0 / (1.0 + np.exp(-z))
def logit(p):  return np.log(p / (1.0 - p))

def signed_diff(target, current):
    """smallest signed angular difference target - current (deg)."""
    return ((target - current + 180) % 360) - 180

def circular_mean(deg):
    r = np.deg2rad(deg)
    return np.degrees(np.arctan2(np.sin(r).sum(), np.cos(r).sum())) % 360

def kernel(kappa):
    """n x n von Mises kernel; row i is a distribution centred on xg[i]."""
    d = np.deg2rad(xg[:, None] - xg[None, :])
    K = np.exp(kappa * np.cos(d))
    return K / K.sum(1, keepdims=True)

def vm_row(mu_deg, kappa):
    d = np.deg2rad(mu_deg - xg)
    v = np.exp(kappa * np.cos(d))
    return v / v.sum()

def response_prob(prior, L, stim_idx, lapse):
    """P(estimate | stimulus) as a length-n vector, marginalizing over measurement."""
    R = (prior[None, :] * L).argmax(1)              # MAP response idx per measurement
    resp = np.bincount(R, weights=L[stim_idx], minlength=n)
    resp = (1 - lapse) * resp / resp.sum() + lapse / n
    return resp

## Peek inside: what `kernel(kappa)` actually is

Before fitting anything, let's *see* the building blocks so the later code is intuitive. `kernel(kappa)` is an `n×n` matrix — row *i* is a von Mises bump centred on grid direction *i*. Two facts to confirm by eye:

- every row is the **same bump, just shifted** (von Mises depends only on the angular difference), and
- the matrix is **symmetric**, which is exactly why one matrix is simultaneously the likelihood (read a column) and `P(measurement | stimulus)` (read a row).

In [ ]:
# @title Visualize the kernel matrix + a few rows
K = kernel(3.0)                      # concentration kappa = 3
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

im = ax[0].imshow(K, origin="lower", extent=[xg[0], xg[-1], xg[0], xg[-1]], aspect="auto")
ax[0].set_title("kernel(3.0): each row a von Mises bump")
ax[0].set_xlabel("evaluated direction (deg)")
ax[0].set_ylabel("center direction (deg)")
fig.colorbar(im, ax=ax[0], shrink=0.8)

for c_deg in [90, 180, 270]:
    i = to_idx(c_deg)
    ax[1].plot(xg, K[i], label=f"row centred {c_deg}\u00b0")
ax[1].set_title("three rows = the same bump, shifted")
ax[1].set_xlabel("direction (deg)"); ax[1].set_ylabel("probability")
ax[1].legend()
plt.tight_layout(); plt.show()

# symmetry / dual-role check: row i vs column i
i = to_idx(180)
print("max |row_i - col_i| =", np.abs(K[i] - K[:, i]).max(),
      "(≈0 -> symmetric -> likelihood = P(meas|stim))")
print("higher kappa = sharper bump:",
      "kappa=1 peak", kernel(1.0)[i].max().round(4),
      "| kappa=8 peak", kernel(8.0)[i].max().round(4))

## Peek inside: the prior `vm_row(225, kappa_prior)`

The prior is one von Mises centred on 225°. `kappa_prior` is its **width knob**: big kappa = narrow, confident belief that pulls estimates hard toward 225°; small kappa = broad, weak belief. This single number is what the fit later estimates from behaviour (`kappa ≈ 1/σ²`).

In [ ]:
# @title Prior shape for several kappa_prior
plt.figure(figsize=(8, 4))
for kp in [0.5, 1, 2, 4, 8]:
    p = vm_row(225, kp)
    plt.plot(xg, p, label=f"kappa_prior={kp}  (\u03c3\u2248{1/np.sqrt(kp):.0f}\u00b0)")
plt.axvline(225, color="gray", ls=":")
plt.title("prior = von Mises at 225 deg; kappa sets confidence")
plt.xlabel("direction (deg)"); plt.ylabel("prior probability")
plt.legend(); plt.tight_layout(); plt.show()

## Peek inside: one Bayesian trial (prior × likelihood → MAP bias)

Bayes rule on the grid: `posterior ∝ prior × likelihood`, and the readout is the MAP (argmax). With the prior at 225°, the posterior peak sits **between** the sensory measurement and 225° — the estimate is *biased toward the prior*. Weakening the sensory reliability (`kappa_llh`) drags it further toward 225°. This is the single-trial version of the whole effect.

In [ ]:
# @title One trial: how the posterior gets pulled toward the prior
meas_deg = 160.0            # this trial's noisy measurement
kp = 2.0                    # prior concentration
prior = vm_row(225, kp)

plt.figure(figsize=(9, 4))
plt.plot(xg, prior, "--", color=[0.75, 0.75, 0], label="prior (225 deg)")
for kllh, col in [(1.0, "tab:blue"), (6.0, "tab:red")]:
    like = vm_row(meas_deg, kllh)                 # likelihood centred on measurement
    post = prior * like; post = post / post.sum() # Bayes: multiply + normalize
    est = xg[post.argmax()]                        # MAP readout
    plt.plot(xg, like, ":", color=col, alpha=0.6)
    plt.plot(xg, post, "-", color=col, label=f"posterior kappa_llh={kllh} -> MAP {est:.0f} deg")
    plt.axvline(est, color=col, ls="-", alpha=0.4)
plt.axvline(meas_deg, color="gray", ls=":"); plt.axvline(225, color="gray", ls=":")
plt.title("dotted=likelihood, solid=posterior; low kappa_llh pulled harder to 225")
plt.xlabel("direction (deg)"); plt.ylabel("probability")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Peek inside: `response_prob` marginalizes the hidden measurement

We never observe the subject's internal measurement — only their final estimate. So the likelihood used for fitting is `P(estimate | stimulus)`, built by summing over every measurement that could have produced each MAP response, then adding a lapse floor. Below: the same stimulus at three sensory reliabilities. Lower `kappa_llh` (lower coherence) ⇒ the response distribution slides toward the 225° prior. That coherence-dependent bias is the experimental signature the fit is chasing.

In [ ]:
# @title P(estimate | stimulus) at three coherences (kappa_llh)
stim_deg = 160.0
s_i = to_idx(stim_deg)
kp = 2.0
prior = vm_row(225, kp)

plt.figure(figsize=(9, 4))
for kllh, lab, col in [(1.5, "low coh", "tab:blue"),
                       (3.0, "mid coh", "tab:green"),
                       (6.0, "high coh", "tab:red")]:
    L = kernel(kllh)
    resp = response_prob(prior, L, s_i, lapse=0.02)
    mean_est = circular_mean(np.repeat(xg, np.round(resp * 10000).astype(int)))
    plt.plot(xg, resp, color=col, label=f"{lab} (kappa_llh={kllh}) -> mean {mean_est:.0f} deg")
plt.axvline(stim_deg, color="k", ls="--", label=f"stimulus {stim_deg:.0f} deg")
plt.axvline(225, color="gray", ls=":", label="prior 225 deg")
plt.title("weaker sensory reliability -> estimate distribution slides toward prior")
plt.xlabel("estimate (deg)"); plt.ylabel("P(estimate | stimulus)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Peek inside: how the *online* prior mean drifts (delta rule)

The online model has no fixed prior — after each trial it nudges the prior mean toward the stimulus just seen: `mu <- mu + LR · signed_diff(stim, mu)`. Small `LR` = sluggish (near-static); large `LR` = the prior chases recent stimuli. This delta rule is the only structural difference from the static model, and it is exactly the update inside `nll_online` below.

In [ ]:
# @title Prior-mean trajectory under the delta rule
rng = np.random.default_rng(1)
stims = (225 + rng.vonmises(0, 4, size=120) * 180 / np.pi) % 360   # fake stimulus stream near 225

plt.figure(figsize=(9, 4))
for lr, col in [(0.02, "tab:blue"), (0.1, "tab:orange"), (0.4, "tab:red")]:
    mu, traj = 225.0, []
    for s in stims:
        mu = (mu + lr * signed_diff(s, mu)) % 360
        traj.append(mu)
    plt.plot(traj, color=col, label=f"LR={lr}")
plt.plot(stims, ".", color="gray", ms=3, alpha=0.5, label="stimuli")
plt.axhline(225, color="k", ls=":")
plt.title("online prior mean: larger LR chases recent stimuli")
plt.xlabel("trial"); plt.ylabel("prior mean mu (deg)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Negative log likelihood for each model

Free parameters (fit in unconstrained space, then transformed back):

| Model | prior each trial | free params |
|---|---|---|
| **static** | `vm(225, kappa_prior)`, fixed | `kappa_llh`×3 coherence, `kappa_prior`, `lapse` |
| **online** | mean drifts toward each stimulus (delta rule, rate `LR`) | static + `LR` |
| **warm-up** | flat until trial `N`, then `vm(circular_mean(first N stimuli), kappa_prior)` | static + `N` (grid-searched) |

There is **no trial feedback** in this task, so the online model updates its prior from the **stimulus** the subject saw.

In [ ]:
# @title NLL functions
def unpack(theta):
    k = np.exp(theta[:3])          # kappa_llh per coherence level
    kp = np.exp(theta[3])          # kappa_prior
    lap = sig(theta[4])            # lapse
    return k, kp, lap

def nll_static(theta, S, E, C):
    k, kp, lap = unpack(theta)
    prior = vm_row(225, kp)
    ll = 0.0
    for c in range(3):
        L = kernel(k[c])
        R = (prior[None, :] * L).argmax(1)
        m = C == c
        for s_i in np.unique(S[m]):                 # cache response dist per unique stimulus
            resp = np.bincount(R, weights=L[s_i], minlength=n)
            resp = (1 - lap) * resp / resp.sum() + lap / n
            e = E[m & (S == s_i)]
            ll += np.log(resp[e] + 1e-12).sum()
    return -ll

def nll_online(theta, S, E, C, Sdeg, order):
    k, kp, lap = unpack(theta)
    lr = sig(theta[5])
    Ls = [kernel(k[c]) for c in range(3)]
    mu = 225.0
    ll = 0.0
    for t in order:
        L = Ls[C[t]]
        prior = vm_row(mu, kp)
        resp = response_prob(prior, L, S[t], lap)
        ll += np.log(resp[E[t]] + 1e-12)
        mu = (mu + lr * signed_diff(Sdeg[t], mu)) % 360   # update from stimulus
    return -ll

def nll_warmup(theta, S, E, C, Sdeg, order, N):
    k, kp, lap = unpack(theta)
    mu = circular_mean(Sdeg[order[:N]])
    Ls = [kernel(k[c]) for c in range(3)]
    prior_fix = vm_row(mu, kp)
    flat = np.ones(n) / n
    Rfix = [(prior_fix[None, :] * Ls[c]).argmax(1) for c in range(3)]
    Rflat = [(flat[None, :] * Ls[c]).argmax(1) for c in range(3)]
    ll = 0.0
    for i, t in enumerate(order):
        L = Ls[C[t]]
        R = Rflat[C[t]] if i < N else Rfix[C[t]]
        resp = np.bincount(R, weights=L[S[t]], minlength=n)
        resp = (1 - lap) * resp / resp.sum() + lap / n
        ll += np.log(resp[E[t]] + 1e-12)
    return -ll

def simulate_static(theta, S, C, rng):
    """generate synthetic estimates from the static model (for parameter recovery)."""
    k, kp, lap = unpack(theta)
    prior = vm_row(225, kp)
    Ls = [kernel(k[c]) for c in range(3)]
    Rs = [(prior[None, :] * Ls[c]).argmax(1) for c in range(3)]
    E = np.zeros(len(S), int)
    for i in range(len(S)):
        c = C[i]
        resp = np.bincount(Rs[c], weights=Ls[c][S[i]], minlength=n)
        resp = (1 - lap) * resp / resp.sum() + lap / n
        E[i] = rng.choice(n, p=resp)
    return E

def fit(nll, x0, args, maxiter=1500):
    return minimize(nll, x0, args=args, method="Nelder-Mead",
                    options={"maxiter": maxiter, "xatol": 1e-2, "fatol": 1e-2})

## Prepare one subject, one prior block

We fit **subject 1**, the **80° prior block** (widest prior → most room for the models to differ), across its three coherences. Trials are put in temporal order for the online / warm-up models. Extend later by looping over subjects and `prior_std`.

In [ ]:
# @title Build arrays for subject 1, prior_std = 80
SUBJECT_ID = 1
PRIOR_STD = 80

sub = data[(data.subject_id == SUBJECT_ID) & (data.prior_std == PRIOR_STD)].copy()
sub = sub.dropna(subset=["estimate_x", "estimate_y"])
sub = sub.sort_values(["session_id", "run_id", "trial_index"]).reset_index(drop=True)

est_deg = np.round(np.degrees(np.arctan2(sub.estimate_y.values, sub.estimate_x.values)) % 360)
Sdeg = sub.motion_direction.values.astype(float)
coh = sub.motion_coherence.values
coh_levels = np.sort(np.unique(coh))          # e.g. [0.06, 0.12, 0.24]

S = to_idx(Sdeg)
E = to_idx(est_deg)
C = np.searchsorted(coh_levels, coh)
order = np.arange(len(S))

print(f"{len(S)} trials | coherences {coh_levels} | unique stimuli {np.unique(Sdeg)}")

## Sanity check: recover known parameters from synthetic data

Before trusting a fit on real data, confirm the fitter can recover parameters it generated itself (Tutorial 3's rule). We simulate estimates from the static model with **known** parameters on subject 1's actual stimulus sequence, then fit and compare.

In [ ]:
# @title Parameter recovery (static model)
rng = np.random.default_rng(0)
true_theta = np.array([np.log(1.5), np.log(3.0), np.log(6.0),  # kappa_llh (rises with coherence)
                       np.log(2.0),                            # kappa_prior
                       logit(0.05)])                           # lapse
E_syn = simulate_static(true_theta, S, C, rng)

x0 = np.array([np.log(1.0), np.log(2.0), np.log(4.0), np.log(1.5), logit(0.1)])
res_syn = fit(nll_static, x0, (S, E_syn, C))

def show(theta):
    k, kp, lap = unpack(theta)
    return f"kllh={np.round(k,2)}  kprior={kp:.2f}  lapse={lap:.3f}"

print("true     :", show(true_theta))
print("recovered:", show(res_syn.x))

## Fit the three models to the real data & compare with AIC

`AIC = 2k + 2·NLL`; lower is better. `k` = number of free parameters (static 5, online 6, warm-up 6 counting the searched `N`).

In [ ]:
# @title Fit static / online / warm-up, then compare
x0s = np.array([np.log(1.0), np.log(2.0), np.log(4.0), np.log(1.5), logit(0.1)])

# static
res_static = fit(nll_static, x0s, (S, E, C))
k_static = 5
aic_static = 2 * k_static + 2 * res_static.fun

# online (warm-start from static, add LR)
x0o = np.append(res_static.x, logit(0.05))
res_online = fit(nll_online, x0o, (S, E, C, Sdeg, order))
k_online = 6
aic_online = 2 * k_online + 2 * res_online.fun

# warm-up: grid-search N, fit static-form params for each
best = None
for N in [10, 20, 40, 80]:
    r = fit(nll_warmup, res_static.x, (S, E, C, Sdeg, order, N))
    if best is None or r.fun < best[1].fun:
        best = (N, r)
N_best, res_warm = best
k_warm = 6
aic_warm = 2 * k_warm + 2 * res_warm.fun

rows = [("static", k_static, res_static.fun, aic_static),
        ("online", k_online, res_online.fun, aic_online),
        (f"warm-up (N={N_best})", k_warm, res_warm.fun, aic_warm)]
print(f"{'model':<18}{'k':>3}{'NLL':>12}{'AIC':>12}")
for name, k, nllv, aic in rows:
    print(f"{name:<18}{k:>3}{nllv:>12.1f}{aic:>12.1f}")
winner = min(rows, key=lambda r: r[3])[0]
print(f"\nlowest AIC -> {winner}")

## Read out and visualize the winning fit

Overlay the model's predicted mean estimate against the subject's data, per stimulus and coherence. A good fit tracks the data's pull toward the 225° prior, stronger at low coherence.

In [ ]:
# @title Predicted vs observed mean estimate (static fit)
k, kp, lap = unpack(res_static.x)
prior = vm_row(225, kp)

plt.figure(figsize=(12, 4))
for c in range(3):
    L = kernel(k[c])
    R = (prior[None, :] * L).argmax(1)
    us = np.unique(S[C == c])
    pred_mean, data_mean, stim_deg = [], [], []
    for s_i in us:
        resp = np.bincount(R, weights=L[s_i], minlength=n)
        resp = (1 - lap) * resp / resp.sum() + lap / n
        pred_mean.append(circular_mean(np.repeat(xg, np.round(resp * 10000).astype(int))))
        e = E[(C == c) & (S == s_i)]
        data_mean.append(circular_mean(xg[e]))
        stim_deg.append(xg[s_i])
    ax = plt.subplot(1, 3, c + 1)
    order_s = np.argsort(stim_deg)
    stim_deg = np.array(stim_deg)[order_s]
    ax.plot(stim_deg, np.array(data_mean)[order_s], "o", color=[0.5, 0, 0], label="data")
    ax.plot(stim_deg, np.array(pred_mean)[order_s], "-", color="k", label="model")
    ax.axhline(225, color="gray", ls=":")
    ax.plot([stim_deg.min(), stim_deg.max()], [stim_deg.min(), stim_deg.max()], "r--", lw=1)
    ax.set_title(f"{int(coh_levels[c]*100)}% coherence")
    ax.set_xlabel("stimulus dir (deg)")
    if c == 0:
        ax.set_ylabel("mean estimate (deg)"); ax.legend()
plt.tight_layout(); plt.show()

## Notes & next steps

- **Scope of this demo.** One subject, one prior block. Loop over `subject_id` and `prior_std` to fit everyone; sum AIC across subjects for a group-level model comparison.
- **Grid & speed.** `GRID_STEP=4` keeps the online fit quick. Drop to 1–2° for the final numbers (slower).
- **Parameter recovery first.** If the synthetic recovery above is poor, tighten the grid or reduce free parameters before reading anything into the real-data fit.
- **Model realism.** `kappa_prior` is currently a single fitted width; the real paper also lets the prior width track the experimental `prior_std`. The online update here is a minimal delta rule on the prior mean; richer versions update the width too.
- **This is the template's core comparison.** Static vs online vs warm-up via AIC = exactly the "online vs static Bayesian" test in the Laquitaine project template (W3D2 Tutorial 3 + W1D2 Model Fitting).